# Fine-tuning mT5 for Coordinate Normalization

This notebook fine-tunes a mT5 sequence-to-sequence model to extract and normalize historical geographic coordinates from raw text into a standardized decimal degree format.

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    EvalPrediction
)
from transformers.trainer_utils import set_seed
from sklearn.model_selection import KFold
import math
set_seed(42)

In [ ]:
df = pd.read_json("../edda_coordinata.json")
df.head()

In [ ]:
input_column = "text"
output_column = "coordinates_txt"

In [ ]:
def normalize_coords(val):
    if not isinstance(val, list):
        return val.replace("[[\'", '').replace('[["', '').replace("\']]", '').replace('"]]', '').replace("], [", ' | ').replace("\'", "'").replace('\\', '')
    return " | ".join([" ".join(inner) for inner in val])

df[output_column] = df["coordinates"].apply(normalize_coords)
df.head()

In [ ]:

df = df[[input_column, output_column]]
data = Dataset.from_pandas(df)
print(data)


In [ ]:
# -----------------------------
# 1. TOKENIZER
# -----------------------------
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# -----------------------------
# 2. TOKENIZATION CHECK
# -----------------------------
sample_input = "* AACH ou ACH, s. f. petite ville d'Allemagne dans le cercle de Souabe, près de la source de l'Aach. Long. 26. 57. lat. 47. 55."
sample_target = "47 55' N 26 57'"

inputs = tokenizer(sample_input, return_tensors="pt")
print("Input IDs:", inputs["input_ids"])
print("Decoded Input:", tokenizer.decode(inputs["input_ids"][0]))

targets = tokenizer(text_target=sample_target, return_tensors="pt")
print("Target IDs:", targets["input_ids"])
print("Decoded Target:", tokenizer.decode(targets["input_ids"][0]))

In [ ]:
# 3. PREPROCESS FUNCTION
# -----------------------------
def preprocess_function(examples):
    inputs = ["extract_coordinates: " + x for x in examples[input_column]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=examples[output_column], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# -----------------------------
# 4. CUSTOM METRICS
# -----------------------------
def compute_metrics(eval_pred):
    if isinstance(eval_pred, EvalPrediction):
        predictions, labels = eval_pred.predictions, eval_pred.label_ids
    else:
        predictions, labels = eval_pred

    # Generated sequences might come as logits, convert if needed
    if predictions.ndim == 3:  # (batch, seq_len, vocab_size)
        predictions = np.argmax(predictions, axis=-1)

    # Replace -100 in labels with the pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Filter out potential invalid token IDs from predictions
    valid_vocab_size = tokenizer.vocab_size
    predictions = np.where((predictions >= 0) & (predictions < valid_vocab_size), predictions, tokenizer.pad_token_id)


    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Exact Match
    exact_matches = [int(p == l) for p, l in zip(decoded_preds, decoded_labels)]
    em_score = np.mean(exact_matches)

    # Character-level F1
    def char_f1(pred, label):
        common = set(pred) & set(label)
        if not common:
            return 0.0
        prec = len(set(pred) & set(label)) / len(set(pred))
        rec = len(set(pred) & set(label)) / len(set(label))
        return 2 * prec * rec / (prec + rec)

    f1_scores = [char_f1(p, l) for p, l in zip(decoded_preds, decoded_labels)]

    return {"exact_match": em_score, "char_f1": np.mean(f1_scores)}

In [ ]:
# -----------------------------
# 5. TRAINING ARGS TEMPLATE
# -----------------------------
num_samples = 4000  # adjust to your dataset size
batch_size = 8
steps_per_epoch = math.ceil(num_samples / batch_size)
num_train_epochs = 10  # upper bound, early stopping will cut earlier
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = min(500, int(0.1 * total_steps))

base_args = dict(
    eval_strategy="steps",
    eval_steps=250,              # evaluate ~twice per epoch (500 steps per epoch)
    save_strategy="steps",
    save_steps=250,
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=1,
    weight_decay=0.01,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="char_f1",
    lr_scheduler_type="linear",
    warmup_steps=warmup_steps,
    save_total_limit=2,
    logging_steps=50,
)

collator = DataCollatorForSeq2Seq(tokenizer)

In [ ]:
# -----------------------------
# 6. K-FOLD CROSS VALIDATION
# -----------------------------

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_metrics = []

for fold, (train_idx, val_idx) in enumerate(kf.split(data)):
    print(f"\n===== Fold {fold} =====")

    train_dataset = data.select(train_idx).map(preprocess_function, batched=True)
    val_dataset = data.select(val_idx).map(preprocess_function, batched=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results/fold_{fold}",
        **base_args
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    metrics = trainer.evaluate()
    print("Fold metrics:", metrics)
    all_metrics.append(metrics)

# -----------------------------
# 7. AGGREGATE RESULTS
# -----------------------------
mean_em = np.mean([m["eval_exact_match"] for m in all_metrics])
mean_f1 = np.mean([m["eval_char_f1"] for m in all_metrics])
print("\n===== Cross-Validation Results =====")
print(f"Mean Exact Match: {mean_em:.4f}")
print(f"Mean Char F1: {mean_f1:.4f}")


In [ ]:
# -----------------------------
# 6. TRAIN ON FULL DATASET AND SAVE
# -----------------------------

print("\n===== Training on Full Dataset and Saving =====")

# Preprocess the entire dataset
full_dataset = data.map(preprocess_function, batched=True)

# Define training arguments for the full dataset
full_training_args = Seq2SeqTrainingArguments(
    output_dir=f"full_model_training", # Save to your drive path
    eval_strategy="no", # No evaluation during full training
    save_strategy="epoch", # Save checkpoint every epoch
    save_total_limit=1, # Keep only the latest checkpoint
    learning_rate=base_args["learning_rate"],
    per_device_train_batch_size=base_args["per_device_train_batch_size"],
    gradient_accumulation_steps=base_args["gradient_accumulation_steps"],
    weight_decay=base_args["weight_decay"],
    num_train_epochs=base_args["num_train_epochs"], # Use the same number of epochs as in cross-validation
    predict_with_generate=True,
    lr_scheduler_type=base_args["lr_scheduler_type"],
    warmup_steps=base_args["warmup_steps"],
    logging_steps=base_args["logging_steps"],
)

# Load a fresh model for training on the full dataset
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a trainer for the full dataset
trainer = Seq2SeqTrainer(
    model=model,
    args=full_training_args,
    train_dataset=full_dataset,
    tokenizer=tokenizer, # Use tokenizer directly as processing_class is deprecated
    data_collator=collator,
    compute_metrics=compute_metrics, # Still include for completeness, though not used without eval
)

# Train the model on the full dataset
trainer.train()


In [ ]:
# Save the final model and tokenizer
model_save_path = f"fine_tuned_mt5_coordinates"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"\nModel and tokenizer saved to: {model_save_path}")

In [ ]:
import os
import zipfile

def zip_folder(folder_path, output_path):
    """Zips the contents of a folder."""
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

# Define the path to the folder to zip
folder_to_zip = f"fine_tuned_mt5_coordinates"

# Define the output path for the zip file
output_zip_file = f"fine_tuned_mt5_coordinates.zip"

# Create the zip file
zip_folder(folder_to_zip, output_zip_file)

print(f"Folder '{folder_to_zip}' successfully zipped to '{output_zip_file}'")

In [ ]:
# -----------------------------
# 8. LOAD MODEL AND MAKE PREDICTION
# -----------------------------

print("\n===== Loading Model and Making Prediction =====")

# Load the fine-tuned model and tokenizer
model_save_path = f"fine_tuned_mt5_coordinates"
loaded_tokenizer = AutoTokenizer.from_pretrained(model_save_path)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(model_save_path)

# Example input text
input_text = "extract_coordinates: * AACH ou ACH, s. f. petite ville d'Allemagne dans le cercle de Souabe, près de la source de l'Aach. Long. 26. 57. lat. 47. 55."

# Tokenize the input text
input_ids = loaded_tokenizer(input_text, return_tensors="pt").input_ids

# Generate prediction
# Set max_length to a reasonable value for the output coordinates
output_ids = loaded_model.generate(input_ids, max_length=128)

# Decode the prediction
predicted_coordinates = loaded_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Input Text: {input_text}")
print(f"Predicted Coordinates: {predicted_coordinates}")

In [ ]:
from transformers import pipeline

# Define the model name on Hugging Face
model_name_on_hub = "GEODE/mt5-small-coords-norm" # Replace 'your_huggingface_username' with your actual username

# Create a text2text-generation pipeline
generator = pipeline("text2text-generation", model=model_name_on_hub)

# Example input text
input_text_for_pipeline = "extract_coordinates: * AACH ou ACH, s. f. petite ville d'Allemagne dans le cercle de Souabe, près de la source de l'Aach. Long. 26. 57. lat. 47. 55."

# Generate prediction using the pipeline
predicted_coordinates_from_pipeline = generator(input_text_for_pipeline, max_length=128)

print(f"Input Text: {input_text_for_pipeline}")
print(f"Predicted Coordinates (from pipeline): {predicted_coordinates_from_pipeline[0]['generated_text']}")